In [0]:
from pyspark.sql import functions as F

# 1. Carregar as dimensões necessárias
df_clinico = spark.table("workspace.default.dim_condicao_clinica")
df_habitos = spark.table("workspace.default.dim_habitos_suplementacao")

# 2. Fazer a junção das informações clínicas e de suplementação pelo id_atleta
df_base = df_clinico.join(df_habitos, on="id_atleta", how="inner")

# 3. Mapear regras de triagem e alertas para revisão farmacêutica
# Regra 1: Alerta sobre uso de medicamentos (potencial interação ou necessidade de ajuste)
df_alerta_med = (
    df_base
    .filter(
        (F.col("usa_medicamento_continuo").rlike("(?i)sim"))
        | (F.col("medicamentos_continuos_detalhe").isNotNull() & (F.trim(F.col("medicamentos_continuos_detalhe")) != ""))
        | (F.col("medicamentos_ocasionais_detalhe").isNotNull() & (F.trim(F.col("medicamentos_ocasionais_detalhe")) != ""))
    )
    .select(
        F.col("id_atleta"),
        F.lit("FARMACO").alias("tipo_alerta"),
        F.lit("ALTO").alias("nivel_prioridade"),
        F.lit("Uso de medicamentos relatado. Necessária avaliação farmacêutica de interações com ergogênicos e nutrientes.").alias("descricao_alerta"),
        F.concat_ws(" | ", 
            F.coalesce(F.col("medicamentos_continuos_detalhe"), F.lit("")),
            F.coalesce(F.col("medicamentos_ocasionais_detalhe"), F.lit(""))
        ).alias("detalhe_origem")
    )
)

# Regra 2: Alerta sobre alergias ou intolerâncias alimentares/farmacológicas
df_alerta_alergia = (
    df_base
    .filter(
        F.col("alergias_intolerancias").isNotNull() 
        & (~F.lower(F.trim(F.col("alergias_intolerancias"))).isin("nao", "não", "nenhuma", "negativo", ""))
    )
    .select(
        F.col("id_atleta"),
        F.lit("ALERGIA_INTOLERANCIA").alias("tipo_alerta"),
        F.lit("CRITICO").alias("nivel_prioridade"),
        F.lit("Alergia ou intolerância informada. Conferir excipientes e alérgenos nas formulações prescritas.").alias("descricao_alerta"),
        F.col("alergias_intolerancias").alias("detalhe_origem")
    )
)

# Regra 3: Alerta sobre sintomas ao exercício ou histórico cardiovascular
df_alerta_cardio = (
    df_base
    .filter(
        (F.col("sintomas_durante_exercicio").isNotNull() & (~F.lower(F.trim(F.col("sintomas_durante_exercicio"))).isin("nao", "não", "nenhum", "")))
        | (F.col("historico_familiar_cardiovascular").rlike("(?i)sim"))
    )
    .select(
        F.col("id_atleta"),
        F.lit("SEGURANCA_CARDIOVASCULAR").alias("tipo_alerta"),
        F.lit("CRITICO").alias("nivel_prioridade"),
        F.lit("Sintoma no exercício ou histórico familiar cardiovascular relatado. Requer cautela com estimulantes e termo de liberação médica.").alias("descricao_alerta"),
        F.concat_ws(" | ",
            F.coalesce(F.col("sintomas_durante_exercicio"), F.lit("")),
            F.coalesce(F.col("sintomas_detalhe"), F.lit(""))
        ).alias("detalhe_origem")
    )
)

# Regra 4: Alerta sobre sensibilidade a estimulantes ou efeito indesejado prévio
df_alerta_estimulantes = (
    df_base
    .filter(
        (F.col("relatou_efeito_indesejado_suplemento").rlike("(?i)sim"))
        | (F.col("reacao_cafeina_estimulantes").rlike("(?i)ruim|mal|ansiedade|taquicardia|palpita|insonia|sensivel"))
    )
    .select(
        F.col("id_atleta"),
        F.lit("SUPLEMENTO_ESTIMULANTE").alias("tipo_alerta"),
        F.lit("MEDIO").alias("nivel_prioridade"),
        F.lit("Histórico de sensibilidade a estimulantes ou evento adverso prévio com suplementos. Evitar doses elevadas de cafeína.").alias("descricao_alerta"),
        F.concat_ws(" | ",
            F.coalesce(F.col("detalhe_efeito_indesejado"), F.lit("")),
            F.coalesce(F.col("reacao_cafeina_estimulantes"), F.lit(""))
        ).alias("detalhe_origem")
    )
)

# 4. Consolidar todos os alertas em um único DataFrame
df_alertas_consolidados = (
    df_alerta_med
    .unionByName(df_alerta_alergia)
    .unionByName(df_alerta_cardio)
    .unionByName(df_alerta_estimulantes)
    .withColumn(
        "id_alerta",
        F.sha2(F.concat_ws("_", F.col("id_atleta"), F.col("tipo_alerta"), F.col("detalhe_origem")), 256)
    )
    .withColumn("status_revisao", F.lit("PENDENTE_REVISAO_FARMACEUTICA"))
    .withColumn("data_avaliacao", F.current_timestamp())
    .select(
        "id_alerta",
        "id_atleta",
        "tipo_alerta",
        "nivel_prioridade",
        "descricao_alerta",
        "detalhe_origem",
        "status_revisao",
        "data_avaliacao"
    )
)

# 5. Salvar na camada Delta como fato_alerta_clinico
(
    df_alertas_consolidados.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.fato_alerta_clinico")
)

print("fato_alerta_clinico criada com sucesso!")
print("Total de alertas gerados na triagem:", spark.table("workspace.default.fato_alerta_clinico").count())

display(spark.table("workspace.default.fato_alerta_clinico"))

fato_alerta_clinico criada com sucesso!
Total de alertas gerados na triagem: 8


id_alerta,id_atleta,tipo_alerta,nivel_prioridade,descricao_alerta,detalhe_origem,status_revisao,data_avaliacao
fe227b8060e39177a74c8f1b121b6bf3bf459c0daa75bde8cce5187f32ba1c52,e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,SEGURANCA_CARDIOVASCULAR,CRITICO,Sintoma no exercício ou histórico familiar cardiovascular relatado. Requer cautela com estimulantes e termo de liberação médica.,Nenhum dos anteriores |,PENDENTE_REVISAO_FARMACEUTICA,2026-09-13T18:24:53.942Z
3fa2837a098a9b8c6a186b3d210a256a05c834db658887f461ff6b3b82b20a5d,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,SEGURANCA_CARDIOVASCULAR,CRITICO,Sintoma no exercício ou histórico familiar cardiovascular relatado. Requer cautela com estimulantes e termo de liberação médica.,Nenhum dos anteriores |,PENDENTE_REVISAO_FARMACEUTICA,2026-09-13T18:24:53.942Z
6f8e235d730f2815abf6f2eda178b04c20ef336e772c3cfd3430f8da3e687743,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,SEGURANCA_CARDIOVASCULAR,CRITICO,Sintoma no exercício ou histórico familiar cardiovascular relatado. Requer cautela com estimulantes e termo de liberação médica.,Nenhum dos anteriores |,PENDENTE_REVISAO_FARMACEUTICA,2026-09-13T18:24:53.942Z
e174b42fc164cd8c9d1a086a2f2444b26f3b0b9090031223d09a8256ebf8e849,e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,FARMACO,ALTO,Uso de medicamentos relatado. Necessária avaliação farmacêutica de interações com ergogênicos e nutrientes.,Nenhum | Nenhum,PENDENTE_REVISAO_FARMACEUTICA,2026-09-13T18:24:53.942Z
3a509a95e48fbe5b7cf421e94599036d6aa7128aba397e458e10dbb43ff94891,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,FARMACO,ALTO,Uso de medicamentos relatado. Necessária avaliação farmacêutica de interações com ergogênicos e nutrientes.,Nenhum | Nenhum,PENDENTE_REVISAO_FARMACEUTICA,2026-09-13T18:24:53.942Z
6826ffd8cc38d0afe2bb53fd17fcdebf909ea75b439af3039cbbdcc28862e8db,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,FARMACO,ALTO,Uso de medicamentos relatado. Necessária avaliação farmacêutica de interações com ergogênicos e nutrientes.,Não há | Não,PENDENTE_REVISAO_FARMACEUTICA,2026-09-13T18:24:53.942Z
57b6dc92ee9de6ea7466fbb8811a142fd5c4df0ff1aefa66c0d1f041dd0a7c27,e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,ALERGIA_INTOLERANCIA,CRITICO,Alergia ou intolerância informada. Conferir excipientes e alérgenos nas formulações prescritas.,Nenhuma conhecida,PENDENTE_REVISAO_FARMACEUTICA,2026-09-13T18:24:53.942Z
75b4bb6556bd2d79fe203e56f164a96e84a802265814aa4dc285f69f9822b58d,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,ALERGIA_INTOLERANCIA,CRITICO,Alergia ou intolerância informada. Conferir excipientes e alérgenos nas formulações prescritas.,Nenhuma conhecida,PENDENTE_REVISAO_FARMACEUTICA,2026-09-13T18:24:53.942Z


In [0]:
# Verificar se as tabelas existem no catálogo workspace.default

print("Tabelas disponíveis no catálogo workspace.default:")

display(
    spark.sql("SHOW TABLES IN workspace.default")
)

Tabelas disponíveis no catálogo workspace.default:


database,tableName,isTemporary
default,bronze_anamnese_raw,false
default,dim_atleta,false
default,dim_atleta_trainingpeaks,false
default,dim_condicao_clinica,false
default,dim_contraindicacao_interacao,false
default,dim_desempenho_corrida,false
default,dim_habitos_suplementacao,false
default,dim_hidratacao_nutricao,false
default,dim_regra_nutricao_treino,false
default,dim_suplemento_catalogo,false


In [0]:
# Verificar individualmente a existência e o número de registros das tabelas

tabelas_para_validar = [
    "workspace.default.fato_alerta_clinico",
    "workspace.default.dim_atleta"
]

for nome_tabela in tabelas_para_validar:
    try:
        quantidade = spark.table(nome_tabela).count()
        print(f"OK | {nome_tabela} | registros: {quantidade}")
    except Exception as erro:
        print(f"ERRO | {nome_tabela}")
        print(str(erro)[:1000])

OK | workspace.default.fato_alerta_clinico | registros: 8
OK | workspace.default.dim_atleta | registros: 3


In [0]:
# CÉLULA 2: Validação de integridade da fato_alerta_clinico (ajustada)
from pyspark.sql import functions as F

# 1. Carregar as tabelas
df_alertas = spark.table("workspace.default.fato_alerta_clinico")
df_atletas = spark.table("workspace.default.dim_atleta")

# 2. Métricas de integridade
total_alertas = df_alertas.count()
atletas_com_alerta = df_alertas.select("id_atleta").distinct().count()

# 3. Verificar alertas sem correspondência na dim_atleta
sem_atleta = df_alertas.join(df_atletas, on="id_atleta", how="left_anti").count()

# 4. Verificar duplicidade de id_alerta
duplicados = (
    df_alertas
    .groupBy("id_alerta")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Total de alertas gerados: {total_alertas}")
print(f"Atletas com alertas: {atletas_com_alerta}")
print(f"Alertas sem atleta na dim_atleta: {sem_atleta}")
print(f"Alertas duplicados: {duplicados}")

# 5. Exibir a tabela consolidada usando apenas o id_atleta (anonimizado)
df_validacao = (
    df_alertas
    .select(
        "id_atleta",
        "tipo_alerta",
        "nivel_prioridade",
        "descricao_alerta",
        "detalhe_origem",
        "status_revisao"
    )
)

display(df_validacao)

Total de alertas gerados: 8
Atletas com alertas: 3
Alertas sem atleta na dim_atleta: 0
Alertas duplicados: 0


id_atleta,tipo_alerta,nivel_prioridade,descricao_alerta,detalhe_origem,status_revisao
e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,SEGURANCA_CARDIOVASCULAR,CRITICO,Sintoma no exercício ou histórico familiar cardiovascular relatado. Requer cautela com estimulantes e termo de liberação médica.,Nenhum dos anteriores |,PENDENTE_REVISAO_FARMACEUTICA
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,SEGURANCA_CARDIOVASCULAR,CRITICO,Sintoma no exercício ou histórico familiar cardiovascular relatado. Requer cautela com estimulantes e termo de liberação médica.,Nenhum dos anteriores |,PENDENTE_REVISAO_FARMACEUTICA
9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,SEGURANCA_CARDIOVASCULAR,CRITICO,Sintoma no exercício ou histórico familiar cardiovascular relatado. Requer cautela com estimulantes e termo de liberação médica.,Nenhum dos anteriores |,PENDENTE_REVISAO_FARMACEUTICA
e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,FARMACO,ALTO,Uso de medicamentos relatado. Necessária avaliação farmacêutica de interações com ergogênicos e nutrientes.,Nenhum | Nenhum,PENDENTE_REVISAO_FARMACEUTICA
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,FARMACO,ALTO,Uso de medicamentos relatado. Necessária avaliação farmacêutica de interações com ergogênicos e nutrientes.,Nenhum | Nenhum,PENDENTE_REVISAO_FARMACEUTICA
9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,FARMACO,ALTO,Uso de medicamentos relatado. Necessária avaliação farmacêutica de interações com ergogênicos e nutrientes.,Não há | Não,PENDENTE_REVISAO_FARMACEUTICA
e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,ALERGIA_INTOLERANCIA,CRITICO,Alergia ou intolerância informada. Conferir excipientes e alérgenos nas formulações prescritas.,Nenhuma conhecida,PENDENTE_REVISAO_FARMACEUTICA
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,ALERGIA_INTOLERANCIA,CRITICO,Alergia ou intolerância informada. Conferir excipientes e alérgenos nas formulações prescritas.,Nenhuma conhecida,PENDENTE_REVISAO_FARMACEUTICA


In [0]:
# CÉLULA 2A: Criar tabela de-para id_atleta -> nome_completo (uso restrito/operacional)
from pyspark.sql import functions as F

# 1. Ler a base bronze original, que ainda contém o nome completo e o e-mail
df_bronze = spark.table("workspace.default.bronze_anamnese_raw")

# 2. Gerar o id_atleta da mesma forma que foi feito nas dimensões (hash do e-mail de login)
df_depara = (
    df_bronze
    .select(
        F.sha2(F.trim(F.lower(F.col("endereco_de_e_mail"))), 256).alias("id_atleta"),
        F.col("nome_completo")
    )
    .dropDuplicates(["id_atleta"])
)

# 3. Salvar como tabela de uso restrito (não deve ser usada em análises agregadas/exportadas)
(
    df_depara.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.dim_atleta_nome_restrito")
)

print("dim_atleta_nome_restrito criada com sucesso.")
display(spark.table("workspace.default.dim_atleta_nome_restrito"))

dim_atleta_nome_restrito criada com sucesso.


id_atleta,nome_completo
e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,Douglas Bordalo da Silva
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,Marcelo Albuquerque de Oliveira
9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza


In [0]:
# CÉLULA 2B: Validação de integridade da fato_alerta_clinico, com nome para visualização operacional
df_alertas = spark.table("workspace.default.fato_alerta_clinico")
df_atletas = spark.table("workspace.default.dim_atleta")
df_nomes = spark.table("workspace.default.dim_atleta_nome_restrito")

total_alertas = df_alertas.count()
atletas_com_alerta = df_alertas.select("id_atleta").distinct().count()
sem_atleta = df_alertas.join(df_atletas, on="id_atleta", how="left_anti").count()
duplicados = (
    df_alertas.groupBy("id_alerta").count().filter(F.col("count") > 1).count()
)

print(f"Total de alertas gerados: {total_alertas}")
print(f"Atletas com alertas: {atletas_com_alerta}")
print(f"Alertas sem atleta na dim_atleta: {sem_atleta}")
print(f"Alertas duplicados: {duplicados}")

df_validacao = (
    df_alertas
    .join(df_nomes, on="id_atleta", how="left")
    .select(
        "nome_completo",
        "id_atleta",
        "tipo_alerta",
        "nivel_prioridade",
        "descricao_alerta",
        "detalhe_origem",
        "status_revisao"
    )
)

display(df_validacao)

Total de alertas gerados: 8
Atletas com alertas: 3
Alertas sem atleta na dim_atleta: 0
Alertas duplicados: 0


nome_completo,id_atleta,tipo_alerta,nivel_prioridade,descricao_alerta,detalhe_origem,status_revisao
Douglas Bordalo da Silva,e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,SEGURANCA_CARDIOVASCULAR,CRITICO,Sintoma no exercício ou histórico familiar cardiovascular relatado. Requer cautela com estimulantes e termo de liberação médica.,Nenhum dos anteriores |,PENDENTE_REVISAO_FARMACEUTICA
Marcelo Albuquerque de Oliveira,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,SEGURANCA_CARDIOVASCULAR,CRITICO,Sintoma no exercício ou histórico familiar cardiovascular relatado. Requer cautela com estimulantes e termo de liberação médica.,Nenhum dos anteriores |,PENDENTE_REVISAO_FARMACEUTICA
Rafael Souto de Souza,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,SEGURANCA_CARDIOVASCULAR,CRITICO,Sintoma no exercício ou histórico familiar cardiovascular relatado. Requer cautela com estimulantes e termo de liberação médica.,Nenhum dos anteriores |,PENDENTE_REVISAO_FARMACEUTICA
Douglas Bordalo da Silva,e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,FARMACO,ALTO,Uso de medicamentos relatado. Necessária avaliação farmacêutica de interações com ergogênicos e nutrientes.,Nenhum | Nenhum,PENDENTE_REVISAO_FARMACEUTICA
Marcelo Albuquerque de Oliveira,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,FARMACO,ALTO,Uso de medicamentos relatado. Necessária avaliação farmacêutica de interações com ergogênicos e nutrientes.,Nenhum | Nenhum,PENDENTE_REVISAO_FARMACEUTICA
Rafael Souto de Souza,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,FARMACO,ALTO,Uso de medicamentos relatado. Necessária avaliação farmacêutica de interações com ergogênicos e nutrientes.,Não há | Não,PENDENTE_REVISAO_FARMACEUTICA
Douglas Bordalo da Silva,e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,ALERGIA_INTOLERANCIA,CRITICO,Alergia ou intolerância informada. Conferir excipientes e alérgenos nas formulações prescritas.,Nenhuma conhecida,PENDENTE_REVISAO_FARMACEUTICA
Marcelo Albuquerque de Oliveira,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,ALERGIA_INTOLERANCIA,CRITICO,Alergia ou intolerância informada. Conferir excipientes e alérgenos nas formulações prescritas.,Nenhuma conhecida,PENDENTE_REVISAO_FARMACEUTICA


In [0]:
# CÉLULA 3: Conferir os schemas antes de criar a tabela Gold

tabelas_gold = [
    "workspace.default.fato_sessao_treino",
    "workspace.default.dim_atleta",
    "workspace.default.dim_desempenho_corrida",
    "workspace.default.dim_habitos_suplementacao",
    "workspace.default.dim_hidratacao_nutricao",
    "workspace.default.fato_alerta_clinico",
    "workspace.default.dim_atleta_nome_restrito"
]

# Exibir as colunas reais de cada tabela
for nome_tabela in tabelas_gold:
    print(f"\n===== {nome_tabela} =====")
    
    try:
        df_tabela = spark.table(nome_tabela)
        for numero, coluna in enumerate(df_tabela.columns, start=1):
            print(numero, "-", coluna)
    except Exception as erro:
        print("ERRO AO LER A TABELA:")
        print(str(erro)[:1000])


===== workspace.default.fato_sessao_treino =====
1 - id_treino
2 - id_atleta
3 - data_treino
4 - modalidade
5 - titulo_treino
6 - descricao_treino
7 - duracao_minutos
8 - distancia_km
9 - fc_media_bpm
10 - fc_max_bpm
11 - tss
12 - id_regra_nutricao

===== workspace.default.dim_atleta =====
1 - id_atleta
2 - data_nascimento
3 - sexo
4 - altura_cm
5 - peso_kg
6 - objetivo_principal
7 - prova_alvo
8 - frequencia_treino_semanal
9 - horario_treino
10 - trainingpeaks_link
11 - data_ingestao
12 - idade_anos

===== workspace.default.dim_desempenho_corrida =====
1 - id_atleta
2 - tempo_5k_bruto
3 - origem_tempo_5k
4 - tempo_10k_bruto
5 - origem_tempo_10k
6 - tempo_5k_segundos
7 - tempo_10k_segundos
8 - tempo_5k_minutos
9 - tempo_10k_minutos
10 - metodo_avaliacao

===== workspace.default.dim_habitos_suplementacao =====
1 - id_atleta
2 - suplementos_atuais
3 - relatou_efeito_indesejado_suplemento
4 - detalhe_efeito_indesejado
5 - reacao_cafeina_estimulantes
6 - consumo_cafe_energeticos
7 - data_

In [0]:
# CÉLULA 5: Tabela de regras de suplementação AGUDA (por sessão de treino)
from pyspark.sql import functions as F

# Cada linha representa uma regra de dose conservadora (piso da faixa),
# a ser aplicada conforme a duração/intensidade da sessão e o peso do atleta
dados_regra_aguda = [
    # (nome_suplemento, criterio_aplicacao, faixa_min, faixa_max, unidade,
    #  formula_calculo, base_calculo, fonte_referencia, observacao_seguranca)

    ("Carboidrato", "Duracao 45-75 min", 0, 0, "g/h",
     "quantidade_fixa:15", "duracao_sessao",
     "ISSN Position Stand - Nutrient Timing (Kerksick et al., 2017)",
     "Pequena quantidade / bochecho de carboidrato. Dose inicial conservadora."),

    ("Carboidrato", "Duracao 75-150 min", 30, 60, "g/h",
     "piso_faixa:30", "duracao_sessao",
     "ACSM/ISSN - Endurance Nutrition Guidelines",
     "Iniciar pelo piso (30g/h) para reduzir risco de desconforto gastrointestinal."),

    ("Carboidrato", "Duracao acima de 150 min", 60, 90, "g/h",
     "piso_faixa:60", "duracao_sessao",
     "ACSM/ISSN - Endurance Nutrition Guidelines",
     "Combinação glicose+frutose recomendada na literatura para doses >60g/h; iniciar pelo piso."),

    ("Cafeina", "Sessoes de intensidade moderada a alta (Z3+)", 3, 6, "mg/kg",
     "peso_kg * 3", "peso_kg",
     "ISSN Position Stand - Caffeine and Exercise Performance (Guest et al., 2021)",
     "Uso do piso da faixa (3mg/kg). Contraindicado se houver alerta de sensibilidade a estimulantes."),

    ("Bicarbonato_de_sodio", "Sessoes de alta intensidade intermitente (ex.: Hyrox, tiros)", 0.2, 0.3, "g/kg",
     "peso_kg * 0.2", "peso_kg",
     "ISSN Position Stand - Beta-Alanine (referência cruzada de tamponamento, Grgic et al., 2021)",
     "Risco de desconforto gastrointestinal em dose alta; iniciar pelo piso e testar previamente em treino, nunca em prova."),

    ("Sodio_eletrolitos", "Treinos/provas longas com sudorese relevante", 300, 700, "mg/h",
     "piso_faixa:300", "taxa_suor_ou_padrao",
     "ACSM - Exercise and Fluid Replacement Position Stand",
     "Usar taxa de suor individual se o atleta souber informar (conhecimento_taxa_suor); senão aplicar o piso padrão."),

    ("Nitrato_beterraba", "Provas ou treinos-chave de endurance", 5, 9, "mmol",
     "piso_faixa:5", "evento_agudo",
     "ISSN Position Stand - Nitrate Supplementation (Jonvik et al., 2022)",
     "Ingestão 2-3h antes do evento. Iniciar pelo piso da faixa."),
]

# CÉLULA 5 CORRIGIDA: Criar tabela de regras de suplementação aguda

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

# Definir explicitamente o schema para evitar conflito entre inteiros e decimais
schema_regra_aguda = StructType([
    StructField("nome_suplemento", StringType(), True),
    StructField("criterio_aplicacao", StringType(), True),
    StructField("faixa_min", DoubleType(), True),
    StructField("faixa_max", DoubleType(), True),
    StructField("unidade", StringType(), True),
    StructField("formula_calculo", StringType(), True),
    StructField("base_calculo", StringType(), True),
    StructField("fonte_referencia", StringType(), True),
    StructField("observacao_seguranca", StringType(), True)
])

# Criar o DataFrame usando o schema explícito
df_regra_aguda = spark.createDataFrame(
    dados_regra_aguda,
    schema=schema_regra_aguda
)

# Criar identificador único e data de criação
df_regra_aguda = (
    df_regra_aguda
    .withColumn(
        "id_regra_suplemento",
        F.sha2(
            F.concat_ws(
                "_",
                F.col("nome_suplemento"),
                F.col("criterio_aplicacao")
            ),
            256
        )
    )
    .withColumn("data_criacao", F.current_timestamp())
)

# Gravar a tabela Delta
(
    df_regra_aguda.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.dim_regra_suplementacao_zona")
)

# Confirmar a criação
tabela_aguda = spark.table(
    "workspace.default.dim_regra_suplementacao_zona"
)

print("dim_regra_suplementacao_zona criada com sucesso.")
print("Total de regras agudas:", tabela_aguda.count())

display(tabela_aguda)

dim_regra_suplementacao_zona criada com sucesso.
Total de regras agudas: 7


nome_suplemento,criterio_aplicacao,faixa_min,faixa_max,unidade,formula_calculo,base_calculo,fonte_referencia,observacao_seguranca,id_regra_suplemento,data_criacao
Carboidrato,Duracao 45-75 min,0.0,0.0,g/h,quantidade_fixa:15,duracao_sessao,"ISSN Position Stand - Nutrient Timing (Kerksick et al., 2017)",Pequena quantidade / bochecho de carboidrato. Dose inicial conservadora.,fe38dc1eb7c2b6aa6515f0538dce06ace5942cd8ca80e9b03cf00dd89341f3bd,2026-09-13T18:53:12.173Z
Carboidrato,Duracao 75-150 min,30.0,60.0,g/h,piso_faixa:30,duracao_sessao,ACSM/ISSN - Endurance Nutrition Guidelines,Iniciar pelo piso (30g/h) para reduzir risco de desconforto gastrointestinal.,d4287287e646039df5bd4bc52b75adf01d12f9ab868defb76ec4353214f134ad,2026-09-13T18:53:12.173Z
Carboidrato,Duracao acima de 150 min,60.0,90.0,g/h,piso_faixa:60,duracao_sessao,ACSM/ISSN - Endurance Nutrition Guidelines,Combinação glicose+frutose recomendada na literatura para doses >60g/h; iniciar pelo piso.,7db273088cde689282278b1246a1c3eb28bcd318e2045d872d79f98da430c3ed,2026-09-13T18:53:12.173Z
Cafeina,Sessoes de intensidade moderada a alta (Z3+),3.0,6.0,mg/kg,peso_kg * 3,peso_kg,"ISSN Position Stand - Caffeine and Exercise Performance (Guest et al., 2021)",Uso do piso da faixa (3mg/kg). Contraindicado se houver alerta de sensibilidade a estimulantes.,600b3e8bf370ee7d7fb9cdf36d9af5e05af6f38f19e95b62e2761ba9a49a310d,2026-09-13T18:53:12.173Z
Bicarbonato_de_sodio,"Sessoes de alta intensidade intermitente (ex.: Hyrox, tiros)",0.2,0.3,g/kg,peso_kg * 0.2,peso_kg,"ISSN Position Stand - Beta-Alanine (referência cruzada de tamponamento, Grgic et al., 2021)","Risco de desconforto gastrointestinal em dose alta; iniciar pelo piso e testar previamente em treino, nunca em prova.",7fe1289bbdfe36ae83beb0abc1178b16beab8b3138539c003a23507dd7fec663,2026-09-13T18:53:12.173Z
Sodio_eletrolitos,Treinos/provas longas com sudorese relevante,300.0,700.0,mg/h,piso_faixa:300,taxa_suor_ou_padrao,ACSM - Exercise and Fluid Replacement Position Stand,Usar taxa de suor individual se o atleta souber informar (conhecimento_taxa_suor); senão aplicar o piso padrão.,292b8a45ce4346261684d36b878782516c3126416e76722e5d2d54ed12ae5d29,2026-09-13T18:53:12.173Z
Nitrato_beterraba,Provas ou treinos-chave de endurance,5.0,9.0,mmol,piso_faixa:5,evento_agudo,"ISSN Position Stand - Nitrate Supplementation (Jonvik et al., 2022)",Ingestão 2-3h antes do evento. Iniciar pelo piso da faixa.,305de0cc90a3798d46d83ce6b303a52fce1f763fcf14685229422b1393533747,2026-09-13T18:53:12.173Z


In [0]:
# CÉLULA 6: Criar tabela de regras de suplementação crônica
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

# Definir as regras de suplementação de uso crônico
dados_regra_cronica = [
    (
        "Beta_alanina",
        3.2,
        6.4,
        "g/dia",
        "piso_faixa:3.2",
        "dose_fixa_diaria",
        "ISSN Position Stand - Beta-Alanine",
        "Uso crônico; iniciar pelo piso da faixa; avaliar parestesia e tolerância individual."
    ),
    (
        "Creatina",
        3.0,
        5.0,
        "g/dia",
        "piso_faixa:3.0",
        "dose_fixa_diaria",
        "ISSN Position Stand - Creatine Supplementation",
        "Dose de manutenção; avaliar contexto clínico e função renal por profissional habilitado."
    ),
    (
        "HMB",
        3.0,
        3.0,
        "g/dia",
        "valor_fixo:3.0",
        "dose_fixa_diaria",
        "ISSN Position Stand - HMB",
        "Uso condicionado à avaliação individual e ao objetivo do atleta."
    ),
    (
        "Omega_3_oleo_de_peixe",
        1.0,
        2.0,
        "g/dia EPA+DHA",
        "piso_faixa:1.0",
        "dose_fixa_diaria",
        "Literatura esportiva sobre EPA e DHA",
        "Considerar uso de anticoagulantes, alergias e orientação profissional."
    ),
    (
        "Proteina_whey",
        1.4,
        2.0,
        "g/kg/dia",
        "peso_kg * 1.4",
        "peso_kg",
        "ISSN Position Stand - Protein and Exercise",
        "Refere-se à proteína total diária, não exclusivamente ao whey."
    ),
    (
        "Glutamina",
        5.0,
        10.0,
        "g/dia",
        "piso_faixa:5.0",
        "dose_fixa_diaria",
        "Literatura esportiva sobre glutamina",
        "Evidência limitada; não deve ser recomendada automaticamente."
    ),
    (
        "Vitamina_D",
        1000.0,
        2000.0,
        "UI/dia",
        "piso_faixa:1000.0",
        "dose_fixa_diaria",
        "Diretrizes gerais de vitamina D",
        "Idealmente depende de avaliação laboratorial."
    ),
    (
        "Magnesio",
        200.0,
        400.0,
        "mg/dia",
        "piso_faixa:200.0",
        "dose_fixa_diaria",
        "Literatura sobre magnésio e função muscular",
        "Avaliar ingestão alimentar, função renal e indicação clínica."
    )
]

# Declarar os tipos explicitamente
schema_regra_cronica = StructType([
    StructField("nome_suplemento", StringType(), True),
    StructField("faixa_min", DoubleType(), True),
    StructField("faixa_max", DoubleType(), True),
    StructField("unidade", StringType(), True),
    StructField("formula_calculo", StringType(), True),
    StructField("base_calculo", StringType(), True),
    StructField("fonte_referencia", StringType(), True),
    StructField("observacao_seguranca", StringType(), True)
])

# Criar o DataFrame com schema explícito
df_regra_cronica = spark.createDataFrame(
    dados_regra_cronica,
    schema=schema_regra_cronica
)

# Criar identificador e data de criação
df_regra_cronica = (
    df_regra_cronica
    .withColumn(
        "id_protocolo_suplemento",
        F.sha2(F.col("nome_suplemento"), 256)
    )
    .withColumn("data_criacao", F.current_timestamp())
)

# Gravar a tabela Delta
(
    df_regra_cronica.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.dim_protocolo_suplementacao_base")
)

# Validar a criação
tabela_cronica = spark.table(
    "workspace.default.dim_protocolo_suplementacao_base"
)

print("dim_protocolo_suplementacao_base criada com sucesso.")
print("Total de regras crônicas:", tabela_cronica.count())

display(tabela_cronica)

dim_protocolo_suplementacao_base criada com sucesso.
Total de regras crônicas: 8


nome_suplemento,faixa_min,faixa_max,unidade,formula_calculo,base_calculo,fonte_referencia,observacao_seguranca,id_protocolo_suplemento,data_criacao
Beta_alanina,3.2,6.4,g/dia,piso_faixa:3.2,dose_fixa_diaria,ISSN Position Stand - Beta-Alanine,Uso crônico; iniciar pelo piso da faixa; avaliar parestesia e tolerância individual.,83f6a1f23a4b978e0712ec54a4d8a751792b085e796f0af0fdd57a36f3d6a5dd,2026-09-13T18:54:19.472Z
Creatina,3.0,5.0,g/dia,piso_faixa:3.0,dose_fixa_diaria,ISSN Position Stand - Creatine Supplementation,Dose de manutenção; avaliar contexto clínico e função renal por profissional habilitado.,cb50b7f3796fc58118d9b8fd492793524a59c4fac70126aa8021dee831274446,2026-09-13T18:54:19.472Z
HMB,3.0,3.0,g/dia,valor_fixo:3.0,dose_fixa_diaria,ISSN Position Stand - HMB,Uso condicionado à avaliação individual e ao objetivo do atleta.,7a0b230e49a788217fce12bcbc46745204b3b0c3707822f7f121cf7daf1cac34,2026-09-13T18:54:19.472Z
Omega_3_oleo_de_peixe,1.0,2.0,g/dia EPA+DHA,piso_faixa:1.0,dose_fixa_diaria,Literatura esportiva sobre EPA e DHA,"Considerar uso de anticoagulantes, alergias e orientação profissional.",82533a83ffb3b4df25f6c9beb14e8045af9db69157d550c89635b2f83e038ace,2026-09-13T18:54:19.472Z
Proteina_whey,1.4,2.0,g/kg/dia,peso_kg * 1.4,peso_kg,ISSN Position Stand - Protein and Exercise,"Refere-se à proteína total diária, não exclusivamente ao whey.",e106a029857a75bcedb4c803299c7ea3f75bbf32ff872bd11ce7571597c4629b,2026-09-13T18:54:19.472Z
Glutamina,5.0,10.0,g/dia,piso_faixa:5.0,dose_fixa_diaria,Literatura esportiva sobre glutamina,Evidência limitada; não deve ser recomendada automaticamente.,7e6577d72f91e4a49265629de4eb0b225db44d042c7a1ad2874c48f6295d127b,2026-09-13T18:54:19.472Z
Vitamina_D,1000.0,2000.0,UI/dia,piso_faixa:1000.0,dose_fixa_diaria,Diretrizes gerais de vitamina D,Idealmente depende de avaliação laboratorial.,b7e1a685e8f174ac4c2391cdc7b8ebe8872a2c576de3132cd4081f7380af1d37,2026-09-13T18:54:19.472Z
Magnesio,200.0,400.0,mg/dia,piso_faixa:200.0,dose_fixa_diaria,Literatura sobre magnésio e função muscular,"Avaliar ingestão alimentar, função renal e indicação clínica.",dfe30b53215a7aa56c9cd0fc2c8730b8d3c689dc16c15a6d6901422cc1e708e4,2026-09-13T18:54:19.472Z


In [0]:
# CÉLULA 7: Reprocessar treinos incluindo tempo nas zonas de frequência cardíaca

from pyspark.sql import functions as F

# Caminho dos arquivos exportados do TrainingPeaks
caminho_volume = "/Volumes/workspace/default/treinos_trainingpeaks/*.csv"

# Ler os arquivos preservando os nomes originais das colunas
df_bruto_zonas = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(caminho_volume)
    .withColumn("caminho_arquivo", F.col("_metadata.file_path"))
)

# Carregar o de-para oficial dos arquivos para os atletas
df_map = spark.table(
    "workspace.default.dim_atleta_trainingpeaks"
).select(
    "id_atleta",
    "termo_busca"
)

# Relacionar cada arquivo ao atleta correspondente
df_com_atleta_zonas = (
    df_bruto_zonas.alias("treino")
    .join(
        df_map.alias("mapa"),
        F.lower(F.col("treino.caminho_arquivo"))
        .contains(F.lower(F.col("mapa.termo_busca"))),
        "inner"
    )
)

# Converter as colunas de zonas para número
colunas_zonas = [
    f"HRZone{i}Minutes"
    for i in range(1, 11)
]

df_zonas_numericas = df_com_atleta_zonas

for coluna in colunas_zonas:
    df_zonas_numericas = df_zonas_numericas.withColumn(
        coluna,
        F.coalesce(
            F.regexp_replace(
                F.col(f"treino.{coluna}"),
                ",",
                "."
            ).cast("double"),
            F.lit(0.0)
        )
    )

# Criar a fato com as métricas originais e as zonas
df_sessoes_zonas = (
    df_zonas_numericas
    .select(
        F.sha2(
            F.concat_ws(
                "_",
                F.col("mapa.id_atleta"),
                F.col("treino.WorkoutDay"),
                F.col("treino.Title")
            ),
            256
        ).alias("id_treino"),

        F.col("mapa.id_atleta").alias("id_atleta"),

        F.col("treino.WorkoutDay")
        .cast("string")
        .alias("data_treino"),

        F.col("treino.WorkoutType").alias("modalidade"),
        F.col("treino.Title").alias("titulo_treino"),
        F.col("treino.WorkoutDescription").alias("descricao_treino"),

        F.when(
            F.col("treino.TimeTotalInHours").cast("double") > 0,
            F.round(
                F.col("treino.TimeTotalInHours").cast("double") * 60,
                1
            )
        )
        .otherwise(
            F.round(
                F.coalesce(
                    F.col("treino.PlannedDuration").cast("double"),
                    F.lit(0.0)
                ) * 60,
                1
            )
        )
        .alias("duracao_minutos"),

        F.when(
            F.col("treino.DistanceInMeters").cast("double") > 0,
            F.round(
                F.col("treino.DistanceInMeters").cast("double") / 1000,
                3
            )
        )
        .otherwise(
            F.round(
                F.coalesce(
                    F.col("treino.PlannedDistanceInMeters").cast("double"),
                    F.lit(0.0)
                ) / 1000,
                3
            )
        )
        .alias("distancia_km"),

        F.col("treino.HeartRateAverage")
        .cast("double")
        .alias("fc_media_bpm"),

        F.col("treino.HeartRateMax")
        .cast("double")
        .alias("fc_max_bpm"),

        F.col("treino.TSS")
        .cast("double")
        .alias("tss"),

        # Zonas de frequência cardíaca do TrainingPeaks
        *[
            F.col(coluna).alias(coluna.lower())
            for coluna in colunas_zonas
        ]
    )
)

# Calcular os agrupamentos de intensidade
df_sessoes_zonas = (
    df_sessoes_zonas
    .withColumn(
        "minutos_zona_baixa",
        F.round(
            F.col("hrzone1minutes") + F.col("hrzone2minutes"),
            2
        )
    )
    .withColumn(
        "minutos_zona_moderada",
        F.round(
            F.col("hrzone3minutes"),
            2
        )
    )
    .withColumn(
        "minutos_zona_alta",
        F.round(
            F.sum(
                F.array(
                    *[
                        F.col(f"hrzone{i}minutes")
                        for i in range(4, 11)
                    ]
                )
            ),
            2
        )
    )
)

In [0]:
# Verificar se a tabela intermediária com zonas foi criada

nome_tabela = "workspace.default.fato_sessao_treino_zonas"

try:
    df_teste = spark.table(nome_tabela)

    print("Tabela encontrada:", nome_tabela)
    print("Quantidade de registros:", df_teste.count())
    print(
        "Atletas distintos:",
        df_teste.select("id_atleta").distinct().count()
    )

    display(df_teste)

except Exception as erro:
    print("A tabela ainda não foi criada.")
    print(str(erro)[:1500])

Tabela encontrada: workspace.default.fato_sessao_treino_zonas
A tabela ainda não foi criada.
[TABLE_OR_VIEW_NOT_FOUND] The table or view `workspace`.`default`.`fato_sessao_treino_zonas` cannot be found. Verify the spelling and correctness of the schema and catalog.
Search path: [`system`.`session`, `system`.`builtin`, `system`.`ai`, `workspace`.`default`].
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;
'Aggregate [unresolvedalias(count(1))]
+- 'UnresolvedRelation [workspace, default, fato_sessao_treino_zonas], [], false


JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.tableOrViewNotFoundWithSearchPath(QueryCompilationErrors.scala:1662)
	at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.tableNotFound(packa

In [0]:
# CÉLULA 7 — Criar a fato_sessao_treino_zonas de forma completa

from pyspark.sql import functions as F

# 1. Definir o caminho dos arquivos CSV do TrainingPeaks
caminho_volume = "/Volumes/workspace/default/treinos_trainingpeaks/*.csv"

# 2. Ler os arquivos brutos como texto para preservar os valores originais
df_bruto = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(caminho_volume)
    .withColumn("caminho_arquivo", F.col("_metadata.file_path"))
)

# 3. Conferir se as colunas de zonas existem nos arquivos
colunas_zonas_originais = [
    f"HRZone{i}Minutes"
    for i in range(1, 11)
]

colunas_ausentes = [
    coluna
    for coluna in colunas_zonas_originais
    if coluna not in df_bruto.columns
]

if colunas_ausentes:
    raise Exception(
        f"Colunas de zonas ausentes no CSV: {colunas_ausentes}"
    )

print("Arquivos lidos com sucesso.")
print("Quantidade de registros brutos:", df_bruto.count())

# 4. Carregar o de-para oficial entre arquivo e atleta
df_mapa = (
    spark.table("workspace.default.dim_atleta_trainingpeaks")
    .select("id_atleta", "termo_busca")
    .filter(
        F.col("id_atleta").isNotNull()
        & F.col("termo_busca").isNotNull()
    )
)

# 5. Relacionar cada arquivo ao atleta correspondente
df_com_atleta = (
    df_bruto.alias("treino")
    .join(
        df_mapa.alias("mapa"),
        F.lower(F.col("treino.caminho_arquivo"))
        .contains(F.lower(F.col("mapa.termo_busca"))),
        "inner"
    )
)

print(
    "Atletas identificados nos arquivos:",
    df_com_atleta.select("mapa.id_atleta").distinct().count()
)

# 6. Converter minutos nas zonas para valores numéricos
df_numerico = df_com_atleta

for numero in range(1, 11):
    coluna_origem = f"treino.HRZone{numero}Minutes"
    coluna_destino = f"hrzone{numero}minutes"

    df_numerico = df_numerico.withColumn(
        coluna_destino,
        F.coalesce(
            F.regexp_replace(
                F.col(coluna_origem),
                ",",
                "."
            ).cast("double"),
            F.lit(0.0)
        )
    )

# 7. Converter métricas numéricas do treino
def numero_coluna(nome_coluna):
    return F.coalesce(
        F.regexp_replace(
            F.col(f"treino.{nome_coluna}"),
            ",",
            "."
        ).cast("double"),
        F.lit(0.0)
    )

df_sessoes = (
    df_numerico
    .select(
        # Identificador determinístico da sessão
        F.sha2(
            F.concat_ws(
                "_",
                F.col("mapa.id_atleta"),
                F.col("treino.WorkoutDay"),
                F.col("treino.Title")
            ),
            256
        ).alias("id_treino"),

        F.col("mapa.id_atleta").alias("id_atleta"),

        F.col("treino.WorkoutDay")
        .cast("string")
        .alias("data_treino"),

        F.col("treino.WorkoutType").alias("modalidade"),
        F.col("treino.Title").alias("titulo_treino"),
        F.col("treino.WorkoutDescription").alias("descricao_treino"),

        # Priorizar duração realizada; usar planejada quando necessário
        F.when(
            numero_coluna("TimeTotalInHours") > 0,
            F.round(numero_coluna("TimeTotalInHours") * 60, 1)
        )
        .otherwise(
            F.round(numero_coluna("PlannedDuration") * 60, 1)
        )
        .alias("duracao_minutos"),

        # Priorizar distância realizada; usar planejada quando necessário
        F.when(
            numero_coluna("DistanceInMeters") > 0,
            F.round(numero_coluna("DistanceInMeters") / 1000, 3)
        )
        .otherwise(
            F.round(numero_coluna("PlannedDistanceInMeters") / 1000, 3)
        )
        .alias("distancia_km"),

        numero_coluna("HeartRateAverage").alias("fc_media_bpm"),
        numero_coluna("HeartRateMax").alias("fc_max_bpm"),
        numero_coluna("TSS").alias("tss"),

        # Zonas individuais
        *[
            F.col(f"hrzone{i}minutes").alias(f"hrzone{i}minutes")
            for i in range(1, 11)
        ]
    )
)

# 8. Calcular os grupos de intensidade por sessão
df_sessoes = (
    df_sessoes
    .withColumn(
        "minutos_zona_baixa",
        F.round(
            F.col("hrzone1minutes") +
            F.col("hrzone2minutes"),
            2
        )
    )
    .withColumn(
        "minutos_zona_moderada",
        F.round(
            F.col("hrzone3minutes"),
            2
        )
    )
    .withColumn(
        "minutos_zona_alta",
        F.round(
            sum(
                [
                    F.col(f"hrzone{i}minutes")
                    for i in range(4, 11)
                ],
                F.lit(0.0)
            ),
            2
        )
    )
)

# 9. Identificar a maior permanência em uma zona
df_sessoes = df_sessoes.withColumn(
    "minutos_zona_maxima",
    F.greatest(
        *[
            F.col(f"hrzone{i}minutes")
            for i in range(1, 11)
        ]
    )
)

# 10. Classificar a zona predominante
df_sessoes = (
    df_sessoes
    .withColumn(
        "zona_fc_predominante",
        F.when(
            F.col("minutos_zona_maxima") <= 0,
            "SEM_ZONA"
        )
        .when(
            F.col("hrzone1minutes") == F.col("minutos_zona_maxima"),
            "Z1"
        )
        .when(
            F.col("hrzone2minutes") == F.col("minutos_zona_maxima"),
            "Z2"
        )
        .when(
            F.col("hrzone3minutes") == F.col("minutos_zona_maxima"),
            "Z3"
        )
        .when(
            F.col("hrzone4minutes") == F.col("minutos_zona_maxima"),
            "Z4"
        )
        .when(
            F.col("hrzone5minutes") == F.col("minutos_zona_maxima"),
            "Z5"
        )
        .when(
            F.col("hrzone6minutes") == F.col("minutos_zona_maxima"),
            "Z6"
        )
        .when(
            F.col("hrzone7minutes") == F.col("minutos_zona_maxima"),
            "Z7"
        )
        .when(
            F.col("hrzone8minutes") == F.col("minutos_zona_maxima"),
            "Z8"
        )
        .when(
            F.col("hrzone9minutes") == F.col("minutos_zona_maxima"),
            "Z9"
        )
        .otherwise("Z10")
    )
    .withColumn(
        "classificacao_intensidade",
        F.when(
            F.col("minutos_zona_alta") >
            F.col("minutos_zona_baixa"),
            "ALTA"
        )
        .when(
            F.col("minutos_zona_moderada") >
            F.col("minutos_zona_baixa"),
            "MODERADA"
        )
        .otherwise("BAIXA")
    )
    .drop("minutos_zona_maxima")
)

# 11. Remover possíveis duplicidades da mesma sessão
df_sessoes = df_sessoes.dropDuplicates(["id_treino"])

# 12. Gravar a tabela intermediária com zonas
(
    df_sessoes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.default.fato_sessao_treino_zonas"
    )
)

# 13. Validar o resultado
df_resultado = spark.table(
    "workspace.default.fato_sessao_treino_zonas"
)

print("fato_sessao_treino_zonas criada com sucesso!")
print("Total de sessões:", df_resultado.count())
print(
    "Atletas distintos:",
    df_resultado.select("id_atleta").distinct().count()
)

display(
    df_resultado.select(
        "id_treino",
        "id_atleta",
        "data_treino",
        "modalidade",
        "titulo_treino",
        "duracao_minutos",
        "distancia_km",
        "hrzone1minutes",
        "hrzone2minutes",
        "hrzone3minutes",
        "hrzone4minutes",
        "hrzone5minutes",
        "minutos_zona_baixa",
        "minutos_zona_moderada",
        "minutos_zona_alta",
        "zona_fc_predominante",
        "classificacao_intensidade"
    )
)

Arquivos lidos com sucesso.
Quantidade de registros brutos: 2
Atletas identificados nos arquivos: 1
fato_sessao_treino_zonas criada com sucesso!
Total de sessões: 2
Atletas distintos: 1


id_treino,id_atleta,data_treino,modalidade,titulo_treino,duracao_minutos,distancia_km,hrzone1minutes,hrzone2minutes,hrzone3minutes,hrzone4minutes,hrzone5minutes,minutos_zona_baixa,minutos_zona_moderada,minutos_zona_alta,zona_fc_predominante,classificacao_intensidade
6b56bdd40e1d6a05e474542d23a905d63ba14e09771165710757cacff5fec45e,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,2026-09-07,Run,Rodagem + final forte,90.0,14.798,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SEM_ZONA,BAIXA
43eff98798fbb78fe16e19662b28632690819a3c289786bf38cbb42f67189bfd,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,2026-09-08,Run,fartlek,30.0,11.983,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SEM_ZONA,BAIXA


In [0]:
# CÉLULA 8: Comparar a fato original com a tabela que contém zonas

from pyspark.sql import functions as F

# Ler as duas tabelas
df_fato_original = spark.table(
    "workspace.default.fato_sessao_treino"
)

df_fato_zonas = spark.table(
    "workspace.default.fato_sessao_treino_zonas"
)

# Contagens gerais
print("Sessões na fato original:", df_fato_original.count())
print("Sessões na fato com zonas:", df_fato_zonas.count())

# Identificar treinos existentes na fato original que não foram encontrados nos CSVs atuais
df_treinos_sem_zonas = (
    df_fato_original
    .select(
        "id_treino",
        "id_atleta",
        "data_treino",
        "titulo_treino"
    )
    .join(
        df_fato_zonas.select("id_treino"),
        on="id_treino",
        how="left_anti"
    )
)

print(
    "Treinos da fato original sem correspondência na tabela de zonas:",
    df_treinos_sem_zonas.count()
)

display(df_treinos_sem_zonas)

# Exibir os treinos que foram encontrados com zonas
display(
    df_fato_zonas.select(
        "id_treino",
        "id_atleta",
        "data_treino",
        "titulo_treino",
        "duracao_minutos",
        "hrzone1minutes",
        "hrzone2minutes",
        "hrzone3minutes",
        "hrzone4minutes",
        "hrzone5minutes",
        "zona_fc_predominante",
        "classificacao_intensidade"
    )
)

Sessões na fato original: 4
Sessões na fato com zonas: 2
Treinos da fato original sem correspondência na tabela de zonas: 2


id_treino,id_atleta,data_treino,titulo_treino
ab11cb42b599b5f2c2b41bbec0d50e2b55099d4d849c4cd82259a40dcd512f6d,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,2026-09-07,Rodagem + final forte
0d6b938e8d495d44ffdf494026896c421162c40899647139869600b555961dda,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,2026-09-08,fartlek


id_treino,id_atleta,data_treino,titulo_treino,duracao_minutos,hrzone1minutes,hrzone2minutes,hrzone3minutes,hrzone4minutes,hrzone5minutes,zona_fc_predominante,classificacao_intensidade
6b56bdd40e1d6a05e474542d23a905d63ba14e09771165710757cacff5fec45e,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,2026-09-07,Rodagem + final forte,90.0,0.0,0.0,0.0,0.0,0.0,SEM_ZONA,BAIXA
43eff98798fbb78fe16e19662b28632690819a3c289786bf38cbb42f67189bfd,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,2026-09-08,fartlek,30.0,0.0,0.0,0.0,0.0,0.0,SEM_ZONA,BAIXA


In [0]:
# CÉLULA 9: Enriquecer a fato_sessao_treino com as zonas sem perder nenhum treino
from pyspark.sql import functions as F

# 1. Carregar a fato original (4 treinos) e a tabela com zonas
df_fato_original = spark.table("workspace.default.fato_sessao_treino")
df_zonas = spark.table("workspace.default.fato_sessao_treino_zonas")

# 2. Selecionar apenas as colunas de zonas e métricas calculadas para o cruzamento
colunas_para_enriquecer = [
    "id_atleta", "data_treino", "titulo_treino",
    "hrzone1minutes", "hrzone2minutes", "hrzone3minutes", "hrzone4minutes", "hrzone5minutes",
    "minutos_zona_baixa", "minutos_zona_moderada", "minutos_zona_alta",
    "zona_fc_predominante", "classificacao_intensidade"
]

df_zonas_resumido = (
    df_zonas
    .select(colunas_para_enriquecer)
    .dropDuplicates(["id_atleta", "data_treino", "titulo_treino"])
)

# 3. Fazer o Left Join a partir da fato original garantindo que os 4 treinos permaneçam
df_fato_enriquecida = (
    df_fato_original.alias("orig")
    .join(
        df_zonas_resumido.alias("zonas"),
        on=["id_atleta", "data_treino", "titulo_treino"],
        how="left"
    )
    .withColumn("zona_fc_predominante", F.coalesce(F.col("zonas.zona_fc_predominante"), F.lit("SEM_ZONA_REGISTRADA")))
    .withColumn("classificacao_intensidade", F.coalesce(F.col("zonas.classificacao_intensidade"), F.lit("NAO_CLASSIFICADA")))
    .withColumn("minutos_zona_baixa", F.coalesce(F.col("zonas.minutos_zona_baixa"), F.lit(0.0)))
    .withColumn("minutos_zona_moderada", F.coalesce(F.col("zonas.minutos_zona_moderada"), F.lit(0.0)))
    .withColumn("minutos_zona_alta", F.coalesce(F.col("zonas.minutos_zona_alta"), F.lit(0.0)))
    .select(
        "orig.*",
        "zona_fc_predominante",
        "classificacao_intensidade",
        "minutos_zona_baixa",
        "minutos_zona_moderada",
        "minutos_zona_alta"
    )
)

# 4. Sobrescrever a fato_sessao_treino com o schema completo
(
    df_fato_enriquecida.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.fato_sessao_treino")
)

# 5. Validação final da fato
df_validacao_final = spark.table("workspace.default.fato_sessao_treino")
print("fato_sessao_treino atualizada com sucesso!")
print("Total de treinos:", df_validacao_final.count())
print("Atletas distintos:", df_validacao_final.select("id_atleta").distinct().count())

display(
    df_validacao_final.select(
        "id_treino", "data_treino", "titulo_treino", "duracao_minutos",
        "zona_fc_predominante", "classificacao_intensidade", "minutos_zona_alta"
    )
)

fato_sessao_treino atualizada com sucesso!
Total de treinos: 4
Atletas distintos: 1


id_treino,data_treino,titulo_treino,duracao_minutos,zona_fc_predominante,classificacao_intensidade,minutos_zona_alta
6b56bdd40e1d6a05e474542d23a905d63ba14e09771165710757cacff5fec45e,2026-09-07,Rodagem + final forte,90.0,SEM_ZONA,BAIXA,0.0
43eff98798fbb78fe16e19662b28632690819a3c289786bf38cbb42f67189bfd,2026-09-08,fartlek,30.0,SEM_ZONA,BAIXA,0.0
ab11cb42b599b5f2c2b41bbec0d50e2b55099d4d849c4cd82259a40dcd512f6d,2026-09-07,Rodagem + final forte,90.0,SEM_ZONA,BAIXA,0.0
0d6b938e8d495d44ffdf494026896c421162c40899647139869600b555961dda,2026-09-08,fartlek,30.0,SEM_ZONA,BAIXA,0.0


In [0]:
# CÉLULA 10: Consolidação da Camada GOLD - Prescrição Diária Personalizada
from pyspark.sql import functions as F

# 1. Carregar as tabelas da camada Silver e Dimensões de Regras
df_treino = spark.table("workspace.default.fato_sessao_treino")
df_atleta = spark.table("workspace.default.dim_atleta")
df_nutricao = spark.table("workspace.default.dim_hidratacao_nutricao")
df_suplementos = spark.table("workspace.default.dim_habitos_suplementacao")
df_alerta = spark.table("workspace.default.fato_alerta_clinico")
df_nomes = spark.table("workspace.default.dim_atleta_nome_restrito")

# 2. Resumir os alertas clínicos por atleta para cruzamento de segurança
df_alerta_resumo = (
    df_alerta
    .groupBy("id_atleta")
    .agg(
        F.count("id_alerta").alias("total_alertas_clinicos"),
        F.max(
            F.when(F.col("nivel_prioridade") == "CRITICO", 3)
             .when(F.col("nivel_prioridade") == "ALTO", 2)
             .when(F.col("nivel_prioridade") == "MEDIO", 1)
             .otherwise(0)
        ).alias("cod_severidade_maxima"),
        F.concat_ws(" | ", F.collect_set("tipo_alerta")).alias("tipos_alertas_ativos"),
        F.max(
            F.when(F.col("tipo_alerta") == "SUPLEMENTO_ESTIMULANTE", F.lit(1))
             .otherwise(F.lit(0))
        ).alias("flag_sensibilidade_estimulante")
    )
    .withColumn(
        "classificacao_risco_clinico",
        F.when(F.col("cod_severidade_maxima") == 3, "RISCO_CRITICO")
         .when(F.col("cod_severidade_maxima") == 2, "RISCO_ALTO")
         .when(F.col("cod_severidade_maxima") == 1, "RISCO_MODERADO")
         .otherwise("RISCO_BAIXO")
    )
)

# 3. Cruzar Treinos com o Perfil do Atleta e Gerar a Prescrição Individualizada
df_gold_prescricao = (
    df_treino.alias("t")
    .join(df_atleta.alias("a"), on="id_atleta", how="inner")
    .join(df_nutricao.alias("n"), on="id_atleta", how="left")
    .join(df_suplementos.alias("s"), on="id_atleta", how="left")
    .join(df_nomes.alias("nom"), on="id_atleta", how="left")
    .join(df_alerta_resumo.alias("al"), on="id_atleta", how="left")
    
    # Preencher nulos de alertas para atletas sem registros clínicos
    .withColumn("total_alertas_clinicos", F.coalesce(F.col("total_alertas_clinicos"), F.lit(0)))
    .withColumn("classificacao_risco_clinico", F.coalesce(F.col("classificacao_risco_clinico"), F.lit("RISCO_BAIXO")))
    .withColumn("flag_sensibilidade_estimulante", F.coalesce(F.col("flag_sensibilidade_estimulante"), F.lit(0)))
    
    # REGRA AGUDA 1: Carboidrato Intra-treino (Dose Piso da Literatura Esportiva)
    .withColumn(
        "prescricao_carboidrato_intra",
        F.when(F.col("t.duracao_minutos") < 45, F.lit("Sem necessidade de reposição intra-treino (apenas hidratação)"))
         .when(F.col("t.duracao_minutos").between(45, 75), F.lit("15g total ou enxágue bucal com solução glicosada"))
         .when(F.col("t.duracao_minutos").between(75, 150), F.lit("30g/h (piso da faixa 30-60g/h - glicose/maltodextrina)"))
         .otherwise(F.lit("60g/h (piso da faixa 60-90g/h - proporção 2:1 glicose:frutose)"))
    )
    
    # REGRA AGUDA 2: Cafeína Pré-treino (Calculada por mg/kg com Trava de Segurança Farmacêutica)
    .withColumn(
        "dose_cafeina_calculada_mg",
        F.round(F.col("a.peso_kg") * 3.0, 0) # Piso conservador de 3 mg/kg
    )
    .withColumn(
        "prescricao_cafeina_pre_treino",
        F.when(F.col("flag_sensibilidade_estimulante") == 1, 
               F.lit("CONTRAINDICADO / BLOQUEADO: Atleta com histórico de sensibilidade/efeito adverso"))
         .when(F.col("t.classificacao_intensidade").isin("ALTA", "MODERADA") | F.lower(F.col("t.titulo_treino")).contains("fartlek"),
               F.concat(F.lit("3 mg/kg pré-treino (~"), F.col("dose_cafeina_calculada_mg").cast("string"), F.lit(" mg) - 45 min antes")))
         .otherwise(F.lit("Opcional / Não essencial para treinos puramente regenerativos/baixa intensidade"))
    )
    
    # REGRA AGUDA 3: Reposição de Sódio / Eletrólitos
    .withColumn(
        "prescricao_sodio_eletrolitos",
        F.when(F.col("t.duracao_minutos") < 60, F.lit("Água pura conforme sede habitual"))
         .when(F.lower(F.coalesce(F.col("n.conhecimento_taxa_suor"), F.lit(""))).contains("sim"),
               F.lit("Reposição individualizada conforme taxa de suor declarada"))
         .otherwise(F.lit("300 mg/h de sódio (piso conservador) em cápsula ou isotônico"))
    )
    
    # REGRA AGUDA 4: Bicarbonato de Sódio para Sessões de Choque/Tamponamento
    .withColumn(
        "prescricao_tamponamento_bicarbonato",
        F.when(F.lower(F.col("t.titulo_treino")).contains("fartlek") | (F.col("t.minutos_zona_alta") > 15),
               F.concat(F.lit("Considerar teste de tolerância: 0.2 g/kg (~"), F.round(F.col("a.peso_kg") * 0.2, 1).cast("string"), F.lit(" g) fracionado 90 min antes")))
         .otherwise(F.lit("Não indicado para a intensidade da sessão"))
    )
    
    # REGRA CRÔNICA: Protocolo de Base do Atleta
    .withColumn(
        "protocolo_cronico_diario",
        F.lit("Creatina: 3g/dia (manutenção) | Beta-Alanina: 3.2g/dia (fracionada) | Ômega-3: 1g/dia EPA+DHA")
    )
    
    # GOVERNANÇA: Status de Liberação Farmacêutica
    .withColumn(
        "parecer_farmaceutico_final",
        F.when(F.col("classificacao_risco_clinico").isin("RISCO_CRITICO", "RISCO_ALTO"), 
               F.lit("PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos"))
         .otherwise(F.lit("LIBERADO: Orientação dentro dos parâmetros seguros da literatura"))
    )
    .withColumn("data_processamento_gold", F.current_timestamp())
    
    # 4. Seleção das Colunas Finais da Camada Gold
    .select(
        F.col("t.id_treino"),
        F.col("t.id_atleta"),
        F.col("nom.nome_completo"),
        F.col("a.peso_kg"),
        F.col("t.data_treino"),
        F.col("t.modalidade"),
        F.col("t.titulo_treino"),
        F.col("t.duracao_minutos"),
        F.col("t.distancia_km"),
        F.col("t.zona_fc_predominante"),
        F.col("t.classificacao_intensidade"),
        F.col("prescricao_carboidrato_intra"),
        F.col("prescricao_cafeina_pre_treino"),
        F.col("prescricao_sodio_eletrolitos"),
        F.col("prescricao_tamponamento_bicarbonato"),
        F.col("protocolo_cronico_diario"),
        F.col("total_alertas_clinicos"),
        F.col("classificacao_risco_clinico"),
        F.col("tipos_alertas_ativos"),
        F.col("parecer_farmaceutico_final"),
        F.col("data_processamento_gold")
    )
)

# 5. Salvar na Tabela Delta Gold Oficial
(
    df_gold_prescricao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_prescricao_diaria")
)

# 6. Exibir o resultado final da prescrição
print("gold_prescricao_diaria consolidada com sucesso!")
print("Total de sessões prescritas:", spark.table("workspace.default.gold_prescricao_diaria").count())

display(
    spark.table("workspace.default.gold_prescricao_diaria").select(
        "nome_completo", "data_treino", "titulo_treino", "duracao_minutos",
        "zona_fc_predominante", "prescricao_carboidrato_intra", 
        "prescricao_cafeina_pre_treino", "prescricao_tamponamento_bicarbonato",
        "parecer_farmaceutico_final"
    )
)

gold_prescricao_diaria consolidada com sucesso!
Total de sessões prescritas: 4


nome_completo,data_treino,titulo_treino,duracao_minutos,zona_fc_predominante,prescricao_carboidrato_intra,prescricao_cafeina_pre_treino,prescricao_tamponamento_bicarbonato,parecer_farmaceutico_final
Marcelo Albuquerque de Oliveira,2026-09-07,Rodagem + final forte,90.0,SEM_ZONA,30g/h (piso da faixa 30-60g/h - glicose/maltodextrina),Opcional / Não essencial para treinos puramente regenerativos/baixa intensidade,Não indicado para a intensidade da sessão,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos
Marcelo Albuquerque de Oliveira,2026-09-08,fartlek,30.0,SEM_ZONA,Sem necessidade de reposição intra-treino (apenas hidratação),3 mg/kg pré-treino (~216.0 mg) - 45 min antes,Considerar teste de tolerância: 0.2 g/kg (~14.4 g) fracionado 90 min antes,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos
Marcelo Albuquerque de Oliveira,2026-09-07,Rodagem + final forte,90.0,SEM_ZONA,30g/h (piso da faixa 30-60g/h - glicose/maltodextrina),Opcional / Não essencial para treinos puramente regenerativos/baixa intensidade,Não indicado para a intensidade da sessão,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos
Marcelo Albuquerque de Oliveira,2026-09-08,fartlek,30.0,SEM_ZONA,Sem necessidade de reposição intra-treino (apenas hidratação),3 mg/kg pré-treino (~216.0 mg) - 45 min antes,Considerar teste de tolerância: 0.2 g/kg (~14.4 g) fracionado 90 min antes,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos


In [0]:
# CÉLULA DE DIAGNÓSTICO: verificar o vínculo entre treino, id_atleta e nome

from pyspark.sql import functions as F

# 1. Carregar as tabelas envolvidas
df_treinos = spark.table(
    "workspace.default.fato_sessao_treino"
)

df_nomes = spark.table(
    "workspace.default.dim_atleta_nome_restrito"
)

df_gold = spark.table(
    "workspace.default.gold_prescricao_diaria"
)

# 2. Mostrar os atletas realmente presentes nos treinos
print("Atletas presentes na fato_sessao_treino:")

display(
    df_treinos
    .groupBy("id_atleta")
    .agg(
        F.count("*").alias("quantidade_treinos"),
        F.collect_set("titulo_treino").alias("treinos")
    )
)

# 3. Mostrar o de-para operacional de nomes
print("De-para id_atleta -> nome:")

display(
    df_nomes.orderBy("nome_completo")
)

# 4. Verificar o nome associado a cada treino
print("Nome associado a cada treino:")

display(
    df_treinos
    .select(
        "id_treino",
        "id_atleta",
        "data_treino",
        "titulo_treino"
    )
    .join(
        df_nomes,
        on="id_atleta",
        how="left"
    )
    .select(
        "id_treino",
        "id_atleta",
        "nome_completo",
        "data_treino",
        "titulo_treino"
    )
    .orderBy("data_treino")
)

# 5. Verificar os nomes efetivamente presentes na Gold
print("Nomes presentes na gold_prescricao_diaria:")

display(
    df_gold
    .groupBy("id_atleta", "nome_completo")
    .count()
    .orderBy("nome_completo")
)

Atletas presentes na fato_sessao_treino:


id_atleta,quantidade_treinos,treinos
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,4,"List(Rodagem + final forte, fartlek)"


De-para id_atleta -> nome:


id_atleta,nome_completo
e19a2a2ee1a48d3b63e6c45ca781e84c898a42554e3cadc94ea2205c27f77b34,Douglas Bordalo da Silva
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,Marcelo Albuquerque de Oliveira
9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza


Nome associado a cada treino:


id_treino,id_atleta,nome_completo,data_treino,titulo_treino
6b56bdd40e1d6a05e474542d23a905d63ba14e09771165710757cacff5fec45e,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,Marcelo Albuquerque de Oliveira,2026-09-07,Rodagem + final forte
ab11cb42b599b5f2c2b41bbec0d50e2b55099d4d849c4cd82259a40dcd512f6d,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,Marcelo Albuquerque de Oliveira,2026-09-07,Rodagem + final forte
43eff98798fbb78fe16e19662b28632690819a3c289786bf38cbb42f67189bfd,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,Marcelo Albuquerque de Oliveira,2026-09-08,fartlek
0d6b938e8d495d44ffdf494026896c421162c40899647139869600b555961dda,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,Marcelo Albuquerque de Oliveira,2026-09-08,fartlek


Nomes presentes na gold_prescricao_diaria:


id_atleta,nome_completo,count
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,Marcelo Albuquerque de Oliveira,4


In [0]:
# CÉLULA: Diagnosticar o de-para dos arquivos do TrainingPeaks

from pyspark.sql import functions as F

# Ler o de-para usado na identificação dos atletas
df_mapa_tp = spark.table(
    "workspace.default.dim_atleta_trainingpeaks"
)

# Ler a tabela protegida com os nomes
df_nomes = spark.table(
    "workspace.default.dim_atleta_nome_restrito"
)

print("De-para atual do TrainingPeaks:")
display(df_mapa_tp)

print("De-para com os nomes dos atletas:")
display(
    df_mapa_tp
    .join(
        df_nomes,
        on="id_atleta",
        how="left"
    )
    .select(
        "id_atleta",
        "nome_completo",
        "termo_busca"
    )
)

De-para atual do TrainingPeaks:


termo_busca,padrao_arquivo,id_atleta
rafael,workouts_rafael_souto_de_souza,ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e


De-para com os nomes dos atletas:


id_atleta,nome_completo,termo_busca
ad7280d24aa18a8ce064d86498981e0d87014ea8ba522cf825f643dfaa3ef62e,Marcelo Albuquerque de Oliveira,rafael


In [0]:
# CÉLULA: Corrigir o atleta associado aos arquivos do Rafael

from pyspark.sql import functions as F

# Obter automaticamente o id_atleta correto do Rafael
df_id_rafael = (
    spark.table("workspace.default.dim_atleta_nome_restrito")
    .filter(
        F.lower(F.col("nome_completo")).contains("rafael")
    )
    .select("id_atleta")
    .limit(1)
)

id_rafael = df_id_rafael.collect()[0]["id_atleta"]

print("ID do Rafael identificado com sucesso:")
print(id_rafael)

# Ler o de-para atual
df_mapa_tp = spark.table(
    "workspace.default.dim_atleta_trainingpeaks"
)

# Corrigir somente a linha cujo termo de busca identifica Rafael.
# As demais associações permanecem inalteradas.
df_mapa_corrigido = (
    df_mapa_tp
    .withColumn(
        "id_atleta",
        F.when(
            F.lower(F.col("termo_busca")).contains("rafael"),
            F.lit(id_rafael)
        )
        .otherwise(F.col("id_atleta"))
    )
)

# Validar o resultado antes de gravar
print("De-para corrigido:")
display(
    df_mapa_corrigido
    .join(
        spark.table(
            "workspace.default.dim_atleta_nome_restrito"
        ),
        on="id_atleta",
        how="left"
    )
    .select(
        "id_atleta",
        "nome_completo",
        "termo_busca"
    )
)

# Sobrescrever a tabela corrigida
(
    df_mapa_corrigido.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.default.dim_atleta_trainingpeaks"
    )
)

print("dim_atleta_trainingpeaks corrigida com sucesso.")

ID do Rafael identificado com sucesso:
9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365
De-para corrigido:


id_atleta,nome_completo,termo_busca
9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza,rafael


dim_atleta_trainingpeaks corrigida com sucesso.


In [0]:
# CÉLULA: Corrigir definitivamente o atleta dos treinos do TrainingPeaks

from pyspark.sql import functions as F

# 1. Carregar as tabelas atuais
df_nomes = spark.table(
    "workspace.default.dim_atleta_nome_restrito"
)

df_fato = spark.table(
    "workspace.default.fato_sessao_treino"
)

df_zonas = spark.table(
    "workspace.default.fato_sessao_treino_zonas"
)

# 2. Localizar o ID correto do Rafael pelo nome protegido
df_rafael = (
    df_nomes
    .filter(
        F.lower(F.col("nome_completo"))
        .contains("rafael souto de souza")
    )
    .select(
        F.col("id_atleta").alias("id_rafael")
    )
    .dropDuplicates()
)

print("Quantidade de IDs encontrados para Rafael:", df_rafael.count())

if df_rafael.count() != 1:
    raise Exception(
        "Não foi possível identificar exatamente um id_atleta para Rafael."
    )

# 3. Atualizar o ID dos treinos da fato original
# Neste projeto, os quatro treinos TrainingPeaks existentes pertencem ao Rafael.
df_fato_corrigida = (
    df_fato
    .crossJoin(df_rafael)
    .drop("id_atleta")
    .withColumnRenamed("id_rafael", "id_atleta")
)

# 4. Atualizar também a tabela intermediária que contém as zonas
df_zonas_corrigida = (
    df_zonas
    .crossJoin(df_rafael)
    .drop("id_atleta")
    .withColumnRenamed("id_rafael", "id_atleta")
)

# 5. Regravar a fato principal
(
    df_fato_corrigida.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.default.fato_sessao_treino"
    )
)

# 6. Regravar a tabela com zonas
(
    df_zonas_corrigida.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.default.fato_sessao_treino_zonas"
    )
)

# 7. Validar o resultado usando a tabela de nomes
df_validacao = (
    spark.table("workspace.default.fato_sessao_treino")
    .join(
        df_nomes,
        on="id_atleta",
        how="left"
    )
)

print("fato_sessao_treino corrigida.")
print("Total de treinos:", df_validacao.count())
print(
    "Atletas distintos:",
    df_validacao.select("id_atleta").distinct().count()
)

display(
    df_validacao.select(
        "id_treino",
        "id_atleta",
        "nome_completo",
        "data_treino",
        "titulo_treino"
    ).orderBy("data_treino")
)

Quantidade de IDs encontrados para Rafael: 1
fato_sessao_treino corrigida.
Total de treinos: 4
Atletas distintos: 1


id_treino,id_atleta,nome_completo,data_treino,titulo_treino
6b56bdd40e1d6a05e474542d23a905d63ba14e09771165710757cacff5fec45e,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza,2026-09-07,Rodagem + final forte
ab11cb42b599b5f2c2b41bbec0d50e2b55099d4d849c4cd82259a40dcd512f6d,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza,2026-09-07,Rodagem + final forte
43eff98798fbb78fe16e19662b28632690819a3c289786bf38cbb42f67189bfd,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza,2026-09-08,fartlek
0d6b938e8d495d44ffdf494026896c421162c40899647139869600b555961dda,9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza,2026-09-08,fartlek


In [0]:
# CÉLULA 10: Consolidação da Camada GOLD - Prescrição Diária Personalizada
from pyspark.sql import functions as F

# 1. Carregar as tabelas da camada Silver e Dimensões de Regras
df_treino = spark.table("workspace.default.fato_sessao_treino")
df_atleta = spark.table("workspace.default.dim_atleta")
df_nutricao = spark.table("workspace.default.dim_hidratacao_nutricao")
df_suplementos = spark.table("workspace.default.dim_habitos_suplementacao")
df_alerta = spark.table("workspace.default.fato_alerta_clinico")
df_nomes = spark.table("workspace.default.dim_atleta_nome_restrito")

# 2. Resumir os alertas clínicos por atleta para cruzamento de segurança
df_alerta_resumo = (
    df_alerta
    .groupBy("id_atleta")
    .agg(
        F.count("id_alerta").alias("total_alertas_clinicos"),
        F.max(
            F.when(F.col("nivel_prioridade") == "CRITICO", 3)
             .when(F.col("nivel_prioridade") == "ALTO", 2)
             .when(F.col("nivel_prioridade") == "MEDIO", 1)
             .otherwise(0)
        ).alias("cod_severidade_maxima"),
        F.concat_ws(" | ", F.collect_set("tipo_alerta")).alias("tipos_alertas_ativos"),
        F.max(
            F.when(F.col("tipo_alerta") == "SUPLEMENTO_ESTIMULANTE", F.lit(1))
             .otherwise(F.lit(0))
        ).alias("flag_sensibilidade_estimulante")
    )
    .withColumn(
        "classificacao_risco_clinico",
        F.when(F.col("cod_severidade_maxima") == 3, "RISCO_CRITICO")
         .when(F.col("cod_severidade_maxima") == 2, "RISCO_ALTO")
         .when(F.col("cod_severidade_maxima") == 1, "RISCO_MODERADO")
         .otherwise("RISCO_BAIXO")
    )
)

# 3. Cruzar Treinos com o Perfil do Atleta e Gerar a Prescrição Individualizada
df_gold_prescricao = (
    df_treino.alias("t")
    .join(df_atleta.alias("a"), on="id_atleta", how="inner")
    .join(df_nutricao.alias("n"), on="id_atleta", how="left")
    .join(df_suplementos.alias("s"), on="id_atleta", how="left")
    .join(df_nomes.alias("nom"), on="id_atleta", how="left")
    .join(df_alerta_resumo.alias("al"), on="id_atleta", how="left")
    
    # Preencher nulos de alertas para atletas sem registros clínicos
    .withColumn("total_alertas_clinicos", F.coalesce(F.col("total_alertas_clinicos"), F.lit(0)))
    .withColumn("classificacao_risco_clinico", F.coalesce(F.col("classificacao_risco_clinico"), F.lit("RISCO_BAIXO")))
    .withColumn("flag_sensibilidade_estimulante", F.coalesce(F.col("flag_sensibilidade_estimulante"), F.lit(0)))
    
    # REGRA AGUDA 1: Carboidrato Intra-treino (Dose Piso da Literatura Esportiva)
    .withColumn(
        "prescricao_carboidrato_intra",
        F.when(F.col("t.duracao_minutos") < 45, F.lit("Sem necessidade de reposição intra-treino (apenas hidratação)"))
         .when(F.col("t.duracao_minutos").between(45, 75), F.lit("15g total ou enxágue bucal com solução glicosada"))
         .when(F.col("t.duracao_minutos").between(75, 150), F.lit("30g/h (piso da faixa 30-60g/h - glicose/maltodextrina)"))
         .otherwise(F.lit("60g/h (piso da faixa 60-90g/h - proporção 2:1 glicose:frutose)"))
    )
    
    # REGRA AGUDA 2: Cafeína Pré-treino (Calculada por mg/kg com Trava de Segurança Farmacêutica)
    .withColumn(
        "dose_cafeina_calculada_mg",
        F.round(F.col("a.peso_kg") * 3.0, 0) # Piso conservador de 3 mg/kg
    )
    .withColumn(
        "prescricao_cafeina_pre_treino",
        F.when(F.col("flag_sensibilidade_estimulante") == 1, 
               F.lit("CONTRAINDICADO / BLOQUEADO: Atleta com histórico de sensibilidade/efeito adverso"))
         .when(F.col("t.classificacao_intensidade").isin("ALTA", "MODERADA") | F.lower(F.col("t.titulo_treino")).contains("fartlek"),
               F.concat(F.lit("3 mg/kg pré-treino (~"), F.col("dose_cafeina_calculada_mg").cast("string"), F.lit(" mg) - 45 min antes")))
         .otherwise(F.lit("Opcional / Não essencial para treinos puramente regenerativos/baixa intensidade"))
    )
    
    # REGRA AGUDA 3: Reposição de Sódio / Eletrólitos
    .withColumn(
        "prescricao_sodio_eletrolitos",
        F.when(F.col("t.duracao_minutos") < 60, F.lit("Água pura conforme sede habitual"))
         .when(F.lower(F.coalesce(F.col("n.conhecimento_taxa_suor"), F.lit(""))).contains("sim"),
               F.lit("Reposição individualizada conforme taxa de suor declarada"))
         .otherwise(F.lit("300 mg/h de sódio (piso conservador) em cápsula ou isotônico"))
    )
    
    # REGRA AGUDA 4: Bicarbonato de Sódio para Sessões de Choque/Tamponamento
    .withColumn(
        "prescricao_tamponamento_bicarbonato",
        F.when(F.lower(F.col("t.titulo_treino")).contains("fartlek") | (F.col("t.minutos_zona_alta") > 15),
               F.concat(F.lit("Considerar teste de tolerância: 0.2 g/kg (~"), F.round(F.col("a.peso_kg") * 0.2, 1).cast("string"), F.lit(" g) fracionado 90 min antes")))
         .otherwise(F.lit("Não indicado para a intensidade da sessão"))
    )
    
    # REGRA CRÔNICA: Protocolo de Base do Atleta
    .withColumn(
        "protocolo_cronico_diario",
        F.lit("Creatina: 3g/dia (manutenção) | Beta-Alanina: 3.2g/dia (fracionada) | Ômega-3: 1g/dia EPA+DHA")
    )
    
    # GOVERNANÇA: Status de Liberação Farmacêutica
    .withColumn(
        "parecer_farmaceutico_final",
        F.when(F.col("classificacao_risco_clinico").isin("RISCO_CRITICO", "RISCO_ALTO"), 
               F.lit("PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos"))
         .otherwise(F.lit("LIBERADO: Orientação dentro dos parâmetros seguros da literatura"))
    )
    .withColumn("data_processamento_gold", F.current_timestamp())
    
    # 4. Seleção das Colunas Finais da Camada Gold
    .select(
        F.col("t.id_treino"),
        F.col("t.id_atleta"),
        F.col("nom.nome_completo"),
        F.col("a.peso_kg"),
        F.col("t.data_treino"),
        F.col("t.modalidade"),
        F.col("t.titulo_treino"),
        F.col("t.duracao_minutos"),
        F.col("t.distancia_km"),
        F.col("t.zona_fc_predominante"),
        F.col("t.classificacao_intensidade"),
        F.col("prescricao_carboidrato_intra"),
        F.col("prescricao_cafeina_pre_treino"),
        F.col("prescricao_sodio_eletrolitos"),
        F.col("prescricao_tamponamento_bicarbonato"),
        F.col("protocolo_cronico_diario"),
        F.col("total_alertas_clinicos"),
        F.col("classificacao_risco_clinico"),
        F.col("tipos_alertas_ativos"),
        F.col("parecer_farmaceutico_final"),
        F.col("data_processamento_gold")
    )
)

# 5. Salvar na Tabela Delta Gold Oficial
(
    df_gold_prescricao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_prescricao_diaria")
)

# 6. Exibir o resultado final da prescrição
print("gold_prescricao_diaria consolidada com sucesso!")
print("Total de sessões prescritas:", spark.table("workspace.default.gold_prescricao_diaria").count())

display(
    spark.table("workspace.default.gold_prescricao_diaria").select(
        "nome_completo", "data_treino", "titulo_treino", "duracao_minutos",
        "zona_fc_predominante", "prescricao_carboidrato_intra", 
        "prescricao_cafeina_pre_treino", "prescricao_tamponamento_bicarbonato",
        "parecer_farmaceutico_final"
    )
)

gold_prescricao_diaria consolidada com sucesso!
Total de sessões prescritas: 4


nome_completo,data_treino,titulo_treino,duracao_minutos,zona_fc_predominante,prescricao_carboidrato_intra,prescricao_cafeina_pre_treino,prescricao_tamponamento_bicarbonato,parecer_farmaceutico_final
Rafael Souto de Souza,2026-09-07,Rodagem + final forte,90.0,SEM_ZONA,30g/h (piso da faixa 30-60g/h - glicose/maltodextrina),Opcional / Não essencial para treinos puramente regenerativos/baixa intensidade,Não indicado para a intensidade da sessão,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos
Rafael Souto de Souza,2026-09-08,fartlek,30.0,SEM_ZONA,Sem necessidade de reposição intra-treino (apenas hidratação),3 mg/kg pré-treino (~246.0 mg) - 45 min antes,Considerar teste de tolerância: 0.2 g/kg (~16.4 g) fracionado 90 min antes,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos
Rafael Souto de Souza,2026-09-07,Rodagem + final forte,90.0,SEM_ZONA,30g/h (piso da faixa 30-60g/h - glicose/maltodextrina),Opcional / Não essencial para treinos puramente regenerativos/baixa intensidade,Não indicado para a intensidade da sessão,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos
Rafael Souto de Souza,2026-09-08,fartlek,30.0,SEM_ZONA,Sem necessidade de reposição intra-treino (apenas hidratação),3 mg/kg pré-treino (~246.0 mg) - 45 min antes,Considerar teste de tolerância: 0.2 g/kg (~16.4 g) fracionado 90 min antes,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos


In [0]:
# Validar o nome final na Gold

from pyspark.sql import functions as F

df_gold_final = spark.table(
    "workspace.default.gold_prescricao_diaria"
)

print("Total de sessões:", df_gold_final.count())

display(
    df_gold_final
    .groupBy("id_atleta", "nome_completo")
    .count()
)

Total de sessões: 4


id_atleta,nome_completo,count
9de211f93f9f2ae5f10ff6e3233ffbbbb120374bc30c47cacf226001b95f6365,Rafael Souto de Souza,4


In [0]:
# CÉLULA 11 (AJUSTADA): Validar ambiente e criar pastas via Python nativo
import os
from pyspark.sql import functions as F

# Usamos subpastas dentro do Volume existente de dados
caminho_base_volume = "/Volumes/workspace/default/treinos_trainingpeaks"
caminho_pendentes = os.path.join(caminho_base_volume, "prescricoes_pendentes")
caminho_aprovadas = os.path.join(caminho_base_volume, "prescricoes_aprovadas")

# Criar as pastas usando a biblioteca padrão do sistema operacional
os.makedirs(caminho_pendentes, exist_ok=True)
os.makedirs(caminho_aprovadas, exist_ok=True)

print("Pastas validadas e prontas:")
print("Pendentes:", caminho_pendentes)
print("Aprovadas:", caminho_aprovadas)

# Validar se a Gold está pronta para leitura
df_gold_validacao = spark.table("workspace.default.gold_prescricao_diaria")
print("Sessões disponíveis na Gold:", df_gold_validacao.count())

# Verificar ReportLab
try:
    import reportlab
    print("Biblioteca ReportLab pronta para uso.")
except ImportError:
    print("ReportLab não encontrado. Rode: %pip install reportlab")

display(
    df_gold_validacao.select(
        "nome_completo", "data_treino", "titulo_treino", 
        "classificacao_intensidade", "parecer_farmaceutico_final"
    ).orderBy("data_treino")
)

Pastas validadas e prontas:
Pendentes: /Volumes/workspace/default/treinos_trainingpeaks/prescricoes_pendentes
Aprovadas: /Volumes/workspace/default/treinos_trainingpeaks/prescricoes_aprovadas
Sessões disponíveis na Gold: 4
ReportLab não encontrado. Rode: %pip install reportlab


nome_completo,data_treino,titulo_treino,classificacao_intensidade,parecer_farmaceutico_final
Rafael Souto de Souza,2026-09-07,Rodagem + final forte,BAIXA,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos
Rafael Souto de Souza,2026-09-07,Rodagem + final forte,BAIXA,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos
Rafael Souto de Souza,2026-09-08,fartlek,BAIXA,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos
Rafael Souto de Souza,2026-09-08,fartlek,BAIXA,PENDENTE DE VALIDACAO CLINICA INDIVIDUAL: Alertas de medicamentos/cardiovasculares ativos
